**Pointing SAPAPI ODS Models to Accenture CDSV: Column Mapping and Transformation**
-  On a high level the process we are going to do is, we will replace the existing source_data cte columns with the accenture columns in the source_data cte. Next, we read the transfor_cte columns from dbt model, check which are missing in accenture columns(technical col) and the ones missing we make them N_NULL AS col_name_, So first we replaced the source_cte columns with accenture columns and then we made the columns which are missing in accenture CDS to _NULL AS col_name_ in transform cte.
- cell 1: Inputs required a) TS_Mapping_Document_ACN_Tables.xlxs (Field mapping document) and b) dbt model for the input CDS. First we read the field mapping document into a dataframe and rename the columns, we only require the accenture columns and its corresponding technical names. (var: mapping_df). We also read the dbt model in a var which will be used later (var: sql).
- cell 2: Using the mapping_df , create the columns for source_data cte. Please change var: housekeeping_fields for _HIST models as per the dbt model.
- cell 3: Replace the source_data cte columns with the new accenture columns.
- cell 4: Parse the transform cte columns from the dbt file.
- cell 5: Compare accenture columns(technical names) with transform_cte columns and get the difference , this diff will be then replaced by _NULL as col_name_.
- cell 6: make missing columns _NULL AS col_name_ and generate the final transform_cte columns to be later replaced.
- cell 7: Replace the transform_data cte columns with the new  modified columns.
- cell 8: Replace the source jinja with new accenture cds lz table name
- cell 9: Write the final updated sql.



In [15]:
import pandas as pd  
import re
from azure.storage.blob import BlobClient, ContainerClient, __version__
import snowflake.connector
import json

In [16]:
#required paths 1)mapping-doc 2)dbt-file
file_path = r"C:\Users\222328\Downloads\TS_Mapping_Document_ACN_Tables.xlsx"
sql_path = r'C:\Sources\Repos\EnterpriseCloud\astro_datautility\dags\dbt\dbt_project\models\ods\sapapi\ODS__SAPAPI__ZCDV_T_CMTU119_AFKO.SQL'

#input CDS and base table
cds = "ZCDV_CM0700_05" 
base_table = "AFKO"
old_lz_table = "ZCDV_T_CMTU119_AFKO"

with open(sql_path,'r') as f:
    sql = f.read()

data = pd.read_excel(file_path, sheet_name=cds, skiprows=1)
mapping_df = data.rename(columns= {f"{cds} fields":"acc_columns",f"{base_table} Fields":"tech_columns"})


In [17]:
#Parse and create the new source cte columns
mapping_df['scrubbed_tech_cols'] = mapping_df['tech_columns'].apply(lambda x: re.sub(r'\(.*\)','',x ))
mapping_df['scrubbed_tech_cols'] = mapping_df['scrubbed_tech_cols'].apply(lambda x: re.sub(r'\s+','',x ))
mapping_df['acc_columns'] = mapping_df['acc_columns'].apply(lambda x: re.sub(r'\s+','',x ))

mapping_df['src_cte_col'] = mapping_df.apply(lambda row: f"JSONFIELD:{row['acc_columns']}::STRING AS {row['scrubbed_tech_cols']}", axis = 1)

housekeeping_fields = ['_FILEDATE',f'_FILENAME']
# housekeeping_fields = ["_FILENAME AS SOURCE_FILENAME",
#        " TO_DATE(substr(_FILENAME,position('.',_FILENAME,1)+1,10),'YYYY-MM-DD') AS SNAPSHOT_DATE",
#       "  _FILEDATE AS LOAD_DATETIME"]

src_cte_col = housekeeping_fields + mapping_df['src_cte_col'].to_list()
src_cte_col_str = ',\n\t\t'.join(src_cte_col)
print(src_cte_col_str)

_FILEDATE,
		_FILENAME,
		JSONFIELD:order_number::STRING AS AUFNR,
		JSONFIELD:basic_start_date::STRING AS GSTRP,
		JSONFIELD:basic_finish_date::STRING AS GLTRP,
		JSONFIELD:scheduled_start::STRING AS GSTRS,
		JSONFIELD:scheduled_start_time::STRING AS GSUZS,
		JSONFIELD:scheduled_finish::STRING AS GLTRS,
		JSONFIELD:scheduled_finish_time::STRING AS GLUZS,
		JSONFIELD:actual_finish_date::STRING AS GLTRI,
		JSONFIELD:actual_start_date::STRING AS GSTRI,
		JSONFIELD:actual_start_time::STRING AS GSUZI,
		JSONFIELD:confirmed_order_finish_date::STRING AS GETRI,
		JSONFIELD:confirmed_order_finish_time::STRING AS GEUZI,
		JSONFIELD:scheduling_type::STRING AS TERKZ,
		JSONFIELD:version_of_available_capacity::STRING AS KAPVERSA,
		JSONFIELD:project_definition::STRING AS PRONR,
		JSONFIELD:indicator_relationships::STRING AS FLG_AOB,
		JSONFIELD:material::STRING AS PLNBEZ,
		JSONFIELD:total_qty::STRING AS GAMNG,
		JSONFIELD:basic_finish_time::STRING AS GLUZP,
		JSONFIELD:basic_start_time::STRING AS

In [18]:
#Compare json and excel columns

json_sql = f'SELECT TOP 1 JSONFIELD FROM LZ_DEV.SAPAPI.{cds}_{base_table}'

conn = snowflake.connector.connect(user= "pritesh.x.gandhi@us.tel.com",
                                    account="telus-global.privatelink", 
                                    client_session_keep_alive = False,
                                    warehouse= "DEVELOPER_WH", 
                                    role= "SYSADMIN_DEV",
                                    threads= 1,    
                                    authenticator="externalbrowser",
                                    autocommit=True)  

cursor = conn.cursor()

cursor.execute(json_sql)
try:
    result = cursor.fetchall()
except Exception as e:
    print(f"FAILED SQL : {e}")

conn.close()

op = json.loads(result[0][0])
json_keys = op.keys()
json_keys = list(json_keys)
doc_keys = mapping_df['acc_columns'].to_list()

print(len(doc_keys),len(json_keys))
diff_1 = set(doc_keys).difference(set(json_keys))
diff_2 = set(json_keys).difference(set(doc_keys))
print(diff_1,diff_2)

81 81
set() set()


In [35]:
doc_keys.sort()==json_keys.sort()

True

In [19]:
#replace only the content inside the SELECT statement in the source_data CTE
pattern = r"with\s+source_data\s+as\s*\(\s*SELECT\s+(.*?)\s+FROM"

replacement = f'''
with source_data as ( 
    SELECT
        {src_cte_col_str}
    FROM 
    '''
updated_sql = re.sub(pattern,replacement, sql, flags=re.DOTALL | re.IGNORECASE)

In [20]:
updated_sql

'/************\n\tModel file generated from SAPAPIUtility \n\t*************/\n\n\t\n\t{{ config(\n    \t  materialized=\'incremental\',\n    \t  incremental_strategy=\'merge\',\n    \t  alias=\'ZCDV_T_CMTU119_AFKO\',\n    \t  unique_key = [\'AUFNR\'],\n    \t  tags=["finance"],\n    \t  transient=false\t\n) }} \n\n\t\nwith source_data as ( \n    SELECT\n        _FILEDATE,\n\t\t_FILENAME,\n\t\tJSONFIELD:order_number::STRING AS AUFNR,\n\t\tJSONFIELD:basic_start_date::STRING AS GSTRP,\n\t\tJSONFIELD:basic_finish_date::STRING AS GLTRP,\n\t\tJSONFIELD:scheduled_start::STRING AS GSTRS,\n\t\tJSONFIELD:scheduled_start_time::STRING AS GSUZS,\n\t\tJSONFIELD:scheduled_finish::STRING AS GLTRS,\n\t\tJSONFIELD:scheduled_finish_time::STRING AS GLUZS,\n\t\tJSONFIELD:actual_finish_date::STRING AS GLTRI,\n\t\tJSONFIELD:actual_start_date::STRING AS GSTRI,\n\t\tJSONFIELD:actual_start_time::STRING AS GSUZI,\n\t\tJSONFIELD:confirmed_order_finish_date::STRING AS GETRI,\n\t\tJSONFIELD:confirmed_order_finish_t

In [21]:
#Parse the transform cte columns from the dbt file
transform_data_cte_cols = re.search(r"transform_data as \(\s*SELECT\s*(.*?)\s*FROM",sql,re.S | re.IGNORECASE)
transform_data_cte_cols = transform_data_cte_cols.group(1)
transform_data_cte_cols = re.sub(r'\s+',' ',transform_data_cte_cols)
transform_data_cte_cols = transform_data_cte_cols.split(', ')
transform_data_cte_cols = [re.sub(r'"','',x,flags=re.IGNORECASE) for x in transform_data_cte_cols]

transform_data_cols = [re.sub(r'(.*)\s+as\s+','',x,flags=re.IGNORECASE) for x in transform_data_cte_cols]
# transform_data_cols = [re.sub(r'"','',x,flags=re.IGNORECASE) for x in transform_data_cols]

In [22]:
#Get the list of columns missing in new accenture CDS to  make it null in transofrm cte
accentr_alias = mapping_df['scrubbed_tech_cols'].to_list()
#accentr_alias = [re.sub(r'\(.*?\)','',x) if '(' in x else x for x in accentr_alias ]

print(len(accentr_alias), len(transform_data_cols))
diff = set(transform_data_cols).difference(set(accentr_alias))
print(diff)
print(len(diff))

to_remove = ['_FILEDATE','_FILENAME','_ROW_HASH','AUDIT_LOG_ID','SOURCE_FILENAME','SNAPSHOT_DATE','LOAD_DATETIME']
make_null = [col for col in diff if col not in to_remove]
print(len(make_null))


81 174
{'NOPCOST', 'SICHZTRM', 'STUFE', 'PVERW', 'REDKZ', 'SAENR', 'CFBDATOFM', 'VORUE', 'IGMNG', 'ATRKZ', 'CFBLZEIH', 'GSUPP', 'OBJTYPE', 'ARSNR', 'KZERB', 'COSTUPD', 'RMZHL', 'ABARB', 'RKNOT', 'CHSCH', 'PAENR', 'BEDID', 'GROID', 'GLUPP', '_FILENAME', 'AENNR', 'PLART', 'KAPTSICHZ', 'STARBID', 'RATID', 'REVLV', 'VFMNG', 'RGEKZ', 'KAPTVORGZ', 'GSTPS', 'VWEGX', '_FILEDATE', 'BOMVERSN', 'LODIV', 'CHPROC', 'FLGAOB', 'LEADAUFNR', 'GLTPS', 'STLTY', 'CSPLIT', 'TERHW', '_ROW_HASH', 'GLTPP', 'AUDIT_LOG_ID', 'ZAEHL', 'PROFID', 'PRODNET', 'ARSPS', 'UPDATE_DATE', 'AUFPT', 'FLGBUNDLE', 'NAUCOST', 'ZKRIZ', 'PLSVN', 'CONFKEY', 'FTRMP', 'FTRPS', 'ADPSP', 'CFBBBDPI', 'FSHMPRODORD', 'MESROUTINGID', 'UPTER', 'PRUEFLOS', 'MILLOCZUSKZ', 'CFBMAXLZ', 'VSNMRV', 'SPLSTAT', 'VORGZTRM', 'FLGARBEI', 'LKNOT', 'ZZAORITTEST', 'RUECK', 'POSNRRMA', 'AUFNT', 'IASMG', 'TRMDT', 'KKALKR', 'CYSEQNR', 'CUOBJ', 'OIHANTYP', 'WEGXX', 'NETZKONT', 'RDKZP', 'RSHID', 'RSNID', 'NODISP', 'GSTPP', 'NTZUE', 'RSNTY', 'GLUPS', 'TRKZP', 

In [23]:
#make missing columns <NULL AS col_name>
for i in range(len(transform_data_cte_cols)):
    if ' AS' in transform_data_cte_cols[i].upper():
        check_alias = re.search(r'(.*)\s+as\s+',transform_data_cte_cols[i],re.IGNORECASE).group(1).strip()
    else:
        check_alias = transform_data_cte_cols[i]
    if check_alias in make_null :
        transform_data_cte_cols[i] = f'NULL AS {check_alias}'
        
transform_data_cols_str = ',\n\t\t'.join(transform_data_cte_cols)

In [24]:
transform_data_cols_str

"_FILEDATE AS _FILEDATE,\n\t\t_FILENAME AS _FILENAME,\n\t\t'{{ invocation_id }}' AS AUDIT_LOG_ID,\n\t\tSHA2('hash_me') AS _ROW_HASH,\n\t\tAUFNR as AUFNR,\n\t\tGLTRP as GLTRP,\n\t\tGSTRP as GSTRP,\n\t\tFTRMS as FTRMS,\n\t\tGLTRS as GLTRS,\n\t\tGSTRS as GSTRS,\n\t\tGSTRI as GSTRI,\n\t\tGETRI as GETRI,\n\t\tGLTRI as GLTRI,\n\t\tFTRMI as FTRMI,\n\t\tNULL AS FTRMP,\n\t\tRSNUM as RSNUM,\n\t\tGASMG as GASMG,\n\t\tGAMNG as GAMNG,\n\t\tGMEIN as GMEIN,\n\t\tPLNBEZ as PLNBEZ,\n\t\tPLNTY as PLNTY,\n\t\tPLNNR as PLNNR,\n\t\tNULL AS PLNAW,\n\t\tPLNAL as PLNAL,\n\t\tNULL AS PVERW,\n\t\tPLAUF as PLAUF,\n\t\tPLSVB as PLSVB,\n\t\tPLNME as PLNME,\n\t\tNULL AS PLSVN,\n\t\tPDATV as PDATV,\n\t\tNULL AS PAENR,\n\t\tNULL AS PLGRP,\n\t\tNULL AS LODIV,\n\t\tNULL AS STLTY,\n\t\tNULL AS STLBEZ,\n\t\tSTLST as STLST,\n\t\tSTLNR as STLNR,\n\t\tSDATV as SDATV,\n\t\tSBMNG as SBMNG,\n\t\tSBMEH as SBMEH,\n\t\tNULL AS SAENR,\n\t\tSTLAL as STLAL,\n\t\tSTLAN as STLAN,\n\t\tSLSVN as SLSVN,\n\t\tSLSBS as SLSBS,\n\t\tAUFLD as

In [25]:
# Replace only the content inside the SELECT statement in the transform cte
pattern = r"transform_data\s+as\s*\(\s*SELECT\s*(.*?)\s*FROM" 

replacement = f'''
transform_data as ( 
    SELECT
        {transform_data_cols_str}
    FROM 
    '''

updated_sql = re.sub(pattern,replacement, updated_sql, flags=re.DOTALL |  re.IGNORECASE)

In [26]:
#replace the source jinja with new accenture cds lz table name
pattern = """\\{\\{ *source *\\( *['"]SAPAPI['"] *, *['"]"""+old_lz_table+"""['"] *\\) *\\}\\}"""
replacement = f"{{{{ source('SAPAPI','{cds}_{base_table}') }}}}"

updated_sql = re.sub(pattern,replacement, updated_sql, flags=re.DOTALL)

In [27]:
#write the file
with open(sql_path,'w') as f:
    f.write(updated_sql)